# Subagents — Claude Managed Agents

Up to now, one agent did all the work in a session. With **multiagent orchestration**, the agent a session runs can hand work to other agents. Each one works in its own **session thread**: an isolated conversation, inside the same sandbox.

```
agents.create(multiagent={...})          →  the lead agent lists the agents it may delegate to
session (primary thread)                 →  the lead agent; your client talks to it, as before
  ├─ thread: Liability Reviewer          →  a subagent, with its own history and model
  └─ thread: Renewal Reviewer            →  another subagent
```

The `multiagent` block offers three ways to hand out work:

| | What happens | Covered in |
|---|---|---|
| **Subagents** | The lead delegates tasks itself, reads each report, and can follow up with the same subagent | **this module** |
| **Dynamic workflows** | The lead writes a program that runs many agents in phases; the server runs it in the background | module 07 |
| **Advisor** | The lead consults a stronger model mid-turn and keeps doing the work itself | (mentioned here) |

We continue with the contract review from module 04. This time a **lead** agent splits the review between two **specialist** subagents and merges their findings.

## 1. Setup

In [1]:
%pip install -q -U "anthropic>=1.13" python-dotenv --user


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

client = anthropic.Anthropic()
environment = client.beta.environments.create(
    name="subagents-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)
print(f"Environment: {environment.id}")

Environment: env_012besEDNsMdDfzZt23M2yvj


## 2. Create the Specialists

**Predefined subagents** are ordinary agents that you create first: each has its own model, system prompt, and tools. These two run on **`claude-haiku-5-5`**, a faster and cheaper model for focused reading tasks, while the lead runs on Opus. Every thread is billed at its own model's rates.

The specialists don't need a `multiagent` block. In fact they can't have one: delegation is **one level deep**, and an agent that has `multiagent` set can't be listed as a subagent.

In [3]:
SPECIALIST_TOOLS = [{
    "type": "agent_toolset_20260401",
    "configs": [{"name": "web_search", "enabled": False}, {"name": "web_fetch", "enabled": False}],
}]

liability_reviewer = client.beta.agents.create(
    name="Liability Reviewer",
    description="Reviews liability caps, indemnities and exclusions in supplier contracts.",
    model="claude-haiku-5-5",
    system=(
        "You review supplier contracts for Brightloom Inc., focusing only on liability: "
        "liability caps, carve-outs from the cap, indemnities, and exclusions of damages. "
        "For each contract, report the cap, what is excluded from it, and one risk for Brightloom. "
        "Quote clause numbers. Be concise."
    ),
    tools=SPECIALIST_TOOLS,
)

renewal_reviewer = client.beta.agents.create(
    name="Renewal Reviewer",
    description="Reviews term, renewal, notice and termination clauses in supplier contracts.",
    model="claude-haiku-5-5",
    system=(
        "You review supplier contracts for Brightloom Inc., focusing only on term and exit: "
        "initial term, automatic renewal, non-renewal notice periods, and termination rights. "
        "For each contract, report the next date by which Brightloom must act to avoid an unwanted renewal, "
        "if any (use the sandbox clock for today's date). Quote clause numbers. Be concise."
    ),
    tools=SPECIALIST_TOOLS,
)

for a in (liability_reviewer, renewal_reviewer):
    print(f"{a.name:<20} {a.id}  v{a.version}  {a.model.id}")

Liability Reviewer   agent_01QQNLPkfuSVHLYzgZdToBuP  v1  claude-haiku-5-5
Renewal Reviewer     agent_013oawYAoDPjVXvxR4awHS8f  v1  claude-haiku-5-5


## 3. Create the Lead with a `multiagent` Block

The lead lists its specialists in **`multiagent.subagents.predefined_agents`**. Three more settings shape what it can do:

- **`subagents.inline_agents`**: besides the agents you list, the lead could define **inline** subagents on the fly. These are unsaved agents with a system prompt the lead writes, running on the lead's own model and tools. They're on by default. We turn them off so that the lead uses only our two specialists.
- **`workflows`**: dynamic workflows are **on by default** with the `multiagent_20261001` type. We turn them off here to look at subagents alone; module 07 is all about them.
- **`advisor`** (off by default) would let the lead consult a stronger model mid-turn. We leave it off.

The references are **pinned**: each listed agent is resolved to its version when the lead is created. If you update a specialist later, update the lead's list to pick up the new version.

Tell the lead **in its system prompt** when to delegate, and to whom. It decides when to delegate; your prompt guides that choice.

> **Names matter.** The lead delegates to a subagent by its **`name`**. Use exactly the names from the system prompt. If they don't match, the lead can't find its subagents and quietly does the work itself.

In [4]:
lead = client.beta.agents.create(
    name="Contract Review Lead",
    model="claude-opus-5-5",
    system=(
        "You lead contract reviews for Brightloom Inc. Contracts are mounted read-only under "
        "/mnt/session/uploads/contracts/. Delegate the liability review to the Liability Reviewer and the "
        "renewal review to the Renewal Reviewer, in parallel. Each specialist can read the files itself; "
        "tell it where they are. Then merge their findings into one memo at /mnt/session/outputs/memo.md "
        "with a short ranked list of actions. Keep chat replies short."
    ),
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "multiagent_20261001",
        "subagents": {
            "type": "enabled",
            "inline_agents": {"type": "disabled"},
            "predefined_agents": [
                {"type": "agent", "id": liability_reviewer.id, "version": liability_reviewer.version},
                {"type": "agent", "id": renewal_reviewer.id, "version": renewal_reviewer.version},
            ],
        },
        "workflows": {"type": "disabled"},
    },
)

print(f"Lead: {lead.id}  v{lead.version}")
print(lead.multiagent)

Lead: agent_01XGacn9zsbjVkYWvRd9tsnv  v1
BetaManagedAgentsMultiagent20261001(advisor=BetaManagedAgentsMultiagentAdvisorDisabled(type='disabled'), subagents=BetaManagedAgentsMultiagentSubagentsEnabled(inline_agents=BetaManagedAgentsMultiagentInlineAgentsDisabled(type='disabled'), predefined_agents=[BetaManagedAgentsAgentReference(id='agent_01QQNLPkfuSVHLYzgZdToBuP', type='agent', version=1), BetaManagedAgentsAgentReference(id='agent_013oawYAoDPjVXvxR4awHS8f', type='agent', version=1)], type='enabled'), type='multiagent_20261001', workflows=BetaManagedAgentsMultiagentWorkflowsDisabled(type='disabled'))


## 4. Mount the Contracts and Start a Session

Same as module 04: upload, then mount through `resources`. **All threads share one sandbox**, so the specialists read the same files the lead sees.

We also set a **budget**. One budget covers every thread in the session, whichever model it runs.

In [5]:
uploaded = {}
for path in sorted(Path("../sample-data/contracts").glob("*.md")):
    uploaded[path.name] = client.files.upload(file=path).id

session = client.beta.sessions.create(
    agent={"type": "agent", "id": lead.id, "version": lead.version},
    environment_id=environment.id,
    title="Contract review with subagents",
    resources=[
        {"type": "file", "file_id": file_id, "mount_path": f"/contracts/{name}"}
        for name, file_id in uploaded.items()
    ],
    budget={"type": "limit", "max_list_cost": {"amount": "300", "currency": "USD"}},
)
print(f"Session: {session.id}  ({len(uploaded)} contracts mounted)")

Session: sesn_01JstdGDCeciajBZp4AjYSKN  (6 contracts mounted)


## 5. Follow the Primary Thread

The session's event stream is the **primary thread**: the lead's own activity, plus a **condensed view** of the subagents:

| Event on the primary stream | Meaning |
|---|---|
| `session.thread_created` | A subagent thread was created (`session_thread_id`, `agent_name`) |
| `agent.thread_message_sent` | The lead sent a task or follow-up to a subagent (`to_agent_name`, `content`) |
| `agent.thread_message_received` | A subagent reported back to the lead (`from_agent_name`, `content`) |
| `session.thread_status_running` / `_idle` / `_terminated` | A thread's status changed; carries `session_thread_id` |

A subagent's own tool calls and messages **stay on its thread's stream**. You see when it starts and what it reports, not every step. Only calls that need *your* client, such as a confirmation or a custom tool, are **cross-posted** to the primary stream, so the module 03 pattern of answering on arrival keeps working.

Mind the two idle events. `session.thread_status_idle` is **one thread** going idle. `session.status_idle` is the **whole session**: it arrives only when no thread is running. Only the second one ends the turn.

In [6]:
def text_of(event) -> str:
    return "".join(block.text for block in event.content if block.type == "text")


def follow(stream) -> str:
    """Print the primary thread's events until the whole session ends its turn."""
    names = {}  # session_thread_id -> agent name
    for event in stream:
        match event.type:
            case "session.thread_created":
                names[event.session_thread_id] = event.agent_name
                print(f"[thread created] {event.agent_name}  ({event.session_thread_id})")
            case "agent.thread_message_sent":
                print(f"[lead -> {event.to_agent_name}] {text_of(event)[:160]}")
            case "agent.thread_message_received":
                print(f"[{event.from_agent_name} -> lead] {text_of(event)[:300]}")
            case "session.thread_status_idle":
                name = names.get(event.session_thread_id, "lead")
                print(f"[thread idle] {name}: {event.stop_reason.type}")
            case "agent.message":
                print(f"Lead: {text_of(event)}")
            case "agent.tool_use":
                print(f"[lead {event.name}] {str(event.input.get('command') or event.input.get('file_path') or '')[:80]}")
            case "session.status_idle":
                if event.stop_reason.type == "requires_action":
                    continue
                print(f"[session idle: {event.stop_reason.type}]")
                return event.stop_reason.type
            case "session.status_terminated":
                return "terminated"


def message(text: str) -> dict:
    return {"type": "user.message", "content": [{"type": "text", "text": text}]}

In [7]:
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[message("Review all the contracts and write the memo.")],
    )
    follow(stream)

[lead bash] ls -la /mnt/session/uploads/contracts/; ls /mnt/session/outputs


[thread created] Liability Reviewer  (sthr_01UuoN1d2MfwDtzMRMJZpugk)
[lead -> Liability Reviewer] You're doing the liability part of a contract review for Brightloom Inc. (we are the customer). There are six supplier contracts, mounted read-only at /mnt/sess


[thread created] Renewal Reviewer  (sthr_011Wf1Cao4DZ7NtEnDjWhcff)
[lead -> Renewal Reviewer] You're doing the term and renewal part of a contract review for Brightloom Inc. (we are the customer). There are six supplier contracts, mounted read-only at /m


Lead: I've started both reviewers on all six contracts. I'll merge what they find into the memo once they report.
[thread idle] lead: end_turn


[Renewal Reviewer -> lead] /mnt/session/work/renewal/report.md. Today is 11 Oct 2026. Halvard Freight is the most urgent: non-renewal notice is due by 2 Nov 2026 (22 days), or the contract auto-renews for a year. Quillon Cloud (4 Jul 2027) is High for lock-in, since convenience exit requires paying all remaining fees. Corvel 


...

[thread idle] Liability Reviewer: end_turn


[lead bash] cat /mnt/session/work/liability/report.md


[lead write] /mnt/session/outputs/memo.md


Lead: I've written the memo, combining both reviews; I checked them against the contracts and recalculated every deadline. Two things need checking with the client before anyone relies on the dates: that no notices have already been sent, and how notice has to be given. Neither is in the files.

Ranked actions:
1. **Halvard Freight:** send written non-renewal notice by **2 Nov 2026** (22 days away), or it renews for a year. Before then, check whether an indemnity schedule exists. The liability clause excludes "indemnification obligations" from the cap, but the contract has no indemnity.
2. **Quillon Cloud:** this is the biggest exposure. It hosts the order-management system, and the only way out early is to pay all remaining fees. Service credits are the only remedy for downtime. Missing the notice deadline of **4 Jul 2027** means a 2-year renewal, so start renegotiating now.
3. **Corvel Analytics:** the cancellation deadline is **12 Feb 2027**. Corvel's liability is capped at 3 months

[thread idle] lead: end_turn
[session idle: end_turn]


Read the trace from top to bottom. The lead creates two threads and sends each a task. The specialists work in parallel, their threads go idle, and each report arrives as an `agent.thread_message_received`. Then the lead merges the reports and writes the memo. The session idles only once every thread is done.

## 6. Inspect the Threads

**`sessions.threads.list()`** returns every thread, including the primary one, which is the one with `parent_thread_id` set to `None`. Each thread carries its own `usage`, priced at its own model. Thread figures are rounded separately and leave out the session's running-time cost, so they don't add up exactly to the session's total.

In [8]:
def cents(amount) -> str:
    return f"${int(amount.amount) / 100:.2f}"


threads = list(client.beta.sessions.threads.list(session.id))
for t in threads:
    role = "primary" if t.parent_thread_id is None else "child"
    name = getattr(t.agent, "name", None) or t.agent.type
    cost = cents(t.usage.list_cost) if t.usage and t.usage.list_cost else "-"
    print(f"{t.id}  {role:<8} {name:<22} {t.status:<10} {cost}")

print(f"\nSession total: {cents(client.beta.sessions.retrieve(session.id).usage.list_cost)}")

sthr_01BRa8dzPFYTNm3vNTvbFv2g  primary  Contract Review Lead   idle       $0.27
sthr_01UuoN1d2MfwDtzMRMJZpugk  child    Liability Reviewer     idle       $0.04
sthr_011Wf1Cao4DZ7NtEnDjWhcff  child    Renewal Reviewer       idle       $0.03

Session total: $0.34


To see what a subagent actually did, **drill into its thread**. `sessions.threads.events.list()` returns the thread's own history: the task it received, its tool calls, and its reply. (`sessions.threads.events.stream()` follows it live.)

In [9]:
liability_thread = next(t for t in threads if getattr(t.agent, "name", "") == "Liability Reviewer")

for event in client.beta.sessions.threads.events.list(liability_thread.id, session_id=session.id):
    match event.type:
        case "agent.thread_message_received":
            print(f"[task from {event.from_agent_name or 'lead'}] {text_of(event)[:200]}")
        case "agent.tool_use":
            print(f"[{event.name}] {str(event.input.get('command') or event.input.get('file_path') or '')[:80]}")
        case "agent.message":
            print(f"Reviewer: {text_of(event)[:300]}")

[task from Contract Review Lead] You're doing the liability part of a contract review for Brightloom Inc. (we are the customer). There are six supplier contracts, mounted read-only at /mnt/session/uploads/contracts/ (01-halvard-freig
[bash] ls -la /mnt/session/uploads/contracts/ && wc -c /mnt/session/uploads/contracts/*
[bash] cd /mnt/session/uploads/contracts/ && for f in *.md; do echo "=== $f"; cat "$f";
[write] /mnt/session/work/liability/report.md
Reviewer: The report is at /mnt/session/work/liability/report.md. It covers all six contracts with ratings: Halvard, Corvel and Quillon are High, Mirefield and Orrin are Medium, and Tessaly is Low. Two items need checking against the full contracts: Halvard's carve-out cites an indemnity clause that doesn't e


## 7. Follow Up with the Same Subagent

A subagent's thread is **persistent**. When the lead sends a follow-up, the same subagent answers with its whole previous review still in context, so it doesn't need to re-read anything. Watch for a **second `agent.thread_message_sent`** to an existing thread, with no new `thread_created`.

In [10]:
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[message(
            "Ask the Liability Reviewer, without re-reading the contracts: which single contract would it "
            "renegotiate first, and what exact clause change would it ask for? Relay its answer."
        )],
    )
    follow(stream)

[lead -> Liability Reviewer] Follow-up question. Please answer from your existing review and don't re-read the contracts. If you renegotiated only one of the six contracts first, which woul


Lead: I've sent the question. I'll pass its answer on when it comes back.
[thread idle] lead: end_turn


[Liability Reviewer -> lead] I would renegotiate Quillon (Cloud Hosting) first. Clause 2.2 makes service credits (capped at 30% of one month's fee) the "sole and exclusive remedy for downtime." If the order-management system goes down for an extended period, Brightloom has no route to recover lost sales or operating costs. That


[thread idle] lead: end_turn


Lead: The Liability Reviewer would renegotiate **Quillon Cloud first**.

**Why:** clause 2.2 makes service credits the "sole and exclusive remedy for downtime," and those credits top out at 30% of one month's fee. If the order-management system is down for a long time, Brightloom can't recover lost sales or extra operating costs. The reviewer sees this as the largest single exposure across the six contracts.

**Proposed new clause 2.2:**
> "2.2 Service credits are Customer's sole and exclusive remedy for downtime of less than [72] consecutive hours in a calendar month. For downtime of [72] or more consecutive hours, or for loss or corruption of Customer data caused by Provider's breach, Customer may recover its direct damages, subject to Section 4."

**Alongside that:** define "data-protection obligations" in clause 4.2, which is currently undefined. The 72-hour threshold is the reviewer's suggestion, not from the contract; Brightloom should set it based on how much downtime it can tol

## 8. Download the Memo, Archive the Threads

The memo is in `/mnt/session/outputs/`, exactly as in module 04.

A session can have **at most 25 child threads at a time**, and idle threads count until they're archived. Archive a subagent's thread once its work is done. (Archiving only works on idle threads; interrupt a running one with `user.interrupt` and its `session_thread_id` first.)

In [11]:
import time
from IPython.display import Markdown, display

memo = None
for _ in range(20):   # outputs can lag the idle by a few seconds
    memo = next((f for f in client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
                 if f.downloadable and f.filename == "memo.md"), None)
    if memo:
        break
    time.sleep(3)

if memo:
    display(Markdown(client.files.download(memo.id).read().decode()))
else:
    print("memo.md not found in the session outputs")

# Memo: Supplier Contract Review (Liability, Term and Renewal)

**To:** Brightloom Inc.
**Date:** 11 October 2026
**Scope:** Six supplier contracts (01–06). This memo covers liability caps, indemnities and exclusions, and term, renewal and termination. It brings together two specialist reviews. I checked their findings against the contract text and recalculated every deadline.

## Ranked actions

1. **Halvard Freight: decide by 2 Nov 2026 (22 days).** If written non-renewal notice under cl. 1.2 is not given by then, the contract renews for a year, to 31 Jan 2028. Before the deadline, confirm whether there is an indemnity schedule. Cl. 4.2 carves "indemnification obligations" out of the cap, but the contract contains no indemnity. If Halvard is staying, use the deadline as leverage to add a cargo-loss and third-party indemnity and a minimum cap for claims early in a term.
2. **Quillon Cloud: start renegotiating now; the notice deadline is 4 Jul 2027.** This hosts the order-management system and is the largest overall exposure. If notice is missed, the contract renews for 2 years. The only convenience exit is paying all remaining fees. Service credits are the sole remedy for downtime, capped at 30% of one month's fee. Ask for: an exit for chronic or total outages, a narrower sole-remedy clause, a defined "data-protection obligations" carve-out (and the DPA, if one exists), a breach indemnity, and a change-of-control right.
3. **Corvel Analytics: cancellation deadline 12 Feb 2027.** Vendor's cap is 3 months' fees, with no carve-outs from the consequential-loss exclusion. Brightloom's own liability is uncapped, and renewal prices can rise without limit. Watch for Vendor's 60-day fee notice, expected around mid-January 2027. Negotiate a data-breach super-cap, an IP indemnity, a mutual cap and a renewal price cap, or cancel.
4. **Mirefield Packaging: plan for expiry on 31 May 2027 (no auto-renewal).** There is no convenience exit, and a shortfall fee applies below 250,000 units a year. Defect remedies are limited to replacement or refund, with a EUR 500k cap and no product-liability or recall indemnity. Begin re-sourcing or renegotiation well before expiry, and ask German counsel to confirm the cap is enforceable.
5. **Orrin Staffing: broaden the indemnity (renewal deadline 1 Aug 2027, low urgency).** The indemnity covers wage-and-hour claims only. It does not cover injury, discrimination, joint-employer or third-party claims, and the contract has no insurance requirement. It is also unclear whether the indemnity sits outside the USD 1M cap; cl. 4.2 says "otherwise", which only implies it. Either party can exit on 30 days' notice, so this can be negotiated at any time.
6. **Tessaly Cleaning: housekeeping.** The contract is month-to-month on 30 days' notice, and Tessaly's liability for property damage is uncapped. Obtain a certificate of insurance, additional-insured status, confirmation of workers' compensation cover, and a bodily-injury indemnity.

**Before relying on any date:** confirm that no notices have already been served, and check the required notice formalities. Neither is in the files.

## Summary by contract

| # | Supplier | Next deadline | Renewal / exit | Liability position | Overall |
|---|---|---|---|---|---|
| 01 | Halvard Freight | **2 Nov 2026** | Auto-renews 1 yr; convenience exit costs 2 months' fees | Mutual 12-month fees-paid cap; indemnity carve-out points to an indemnity that doesn't exist; no consequential-loss exclusion | **High, urgent** |
| 05 | Quillon Cloud | 4 Jul 2027 | Auto-renews **2 yrs**; exit means paying remaining fees | Provider cap 6 months' fees; credits sole downtime remedy; undefined data carve-out | **High** |
| 02 | Corvel Analytics | 12 Feb 2027 | Auto-renews 1 yr; uncapped price rises; change-of-control exit | Vendor cap 3 months' fees; no carve-outs; Customer uncapped | **High** |
| 03 | Mirefield Packaging | Expires 31 May 2027 | No renewal; no convenience exit; volume minimum | Seller cap EUR 500k; replace/refund only; no indemnity | Medium |
| 06 | Orrin Staffing | 1 Aug 2027 | Auto-renews 1 yr; 30-day convenience exit | Mutual USD 1M cap; narrow wage-and-hour indemnity | Medium |
| 04 | Tessaly Cleaning | None (month-to-month) | 30-day exit, no penalty | Uncapped for property damage; USD 2M insurance; no injury indemnity | Low |

## Deadline method

No contract defines when a term ends. Where "N years from the Effective Date" could mean the day before the anniversary or the anniversary itself, the earlier reading is used, which moves each deadline one day earlier. Calculations:

- Halvard: 31 Jan 2027 − 90 days = 2 Nov 2026
- Corvel: 14 Mar 2027 − 30 days = 12 Feb 2027
- Quillon: 31 Dec 2027 − 180 days = 4 Jul 2027
- Orrin: 30 Sep 2027 − 60 days = 1 Aug 2027

## Open items and caveats

- Halvard: look for a missing indemnity schedule or amendment (cl. 4.2).
- Quillon: get the full agreement and any DPA that defines "data-protection obligations" (cl. 4.2).
- This review is based on the text only. Counsel should confirm enforceability under each governing law (NY, CA, DE, IL, TX, and Germany with Hamburg courts).

*Detailed working reports: /mnt/session/work/liability/report.md and /mnt/session/work/renewal/report.md.*


In [12]:
for t in client.beta.sessions.threads.list(session.id):
    if t.parent_thread_id is not None and t.status == "idle":
        archived = client.beta.sessions.threads.archive(t.id, session_id=session.id)
        print(f"Archived {getattr(t.agent, 'name', t.agent.type)}: {archived.status}")

for file_id in uploaded.values():
    client.files.delete(file_id)

Archived Liability Reviewer: terminated


Archived Renewal Reviewer: terminated


## Summary

```
specialists              →  ordinary agents (own model, prompt, tools); no multiagent block
lead.multiagent          →  {type: multiagent_20261001,
                              subagents: {type: enabled, predefined_agents: [...], inline_agents: ...},
                              workflows: {type: disabled}}
primary stream           →  thread_created, thread_message_sent / _received, thread_status_*
session.status_idle      →  only when NO thread is running
threads.list / threads.events.list / threads.archive
```

Key takeaways:
- Each subagent runs in its own **thread**: isolated history, shared sandbox, shared budget, its own model
- The primary stream is a **condensed view**; drill into a thread for the details
- Threads persist, so the lead can follow up with the same subagent
- Delegation is one level deep, with at most 25 child threads at a time (archive idle ones)
- Calls that need your client are cross-posted to the primary stream, so answering on arrival still works

Subagents are good for a few specialists that the lead talks to. For **many** pieces of work, such as one agent per contract across hundreds of contracts or a review of every result, the lead would spend its whole context relaying messages. Module 07 hands that coordination to a **program** instead.

### Next steps
- **07-dynamic-workflows** — let the agent write a workflow that runs many agents in parallel